# 课后练习解答（04.03_pyacl_om_inference）

本解答对应《PyACL 加载 OM 模型端侧推理》课后练习，共 15 题。


### 问题1（单选题）

**题目：** PyACL 推理流程中，最先需要完成的运行时步骤通常是？

A. acl.init 并设置 device
B. 读取输出 buffer
C. 执行 CPU NMS
D. 打开 MindStudio

**解答：** A

**解析：** ACL 运行时必须先初始化并设置设备，后续才能加载模型和分配内存。


### 问题2（单选题）

**题目：** `acl.mdl.load_from_file` 的主要作用是？

A. 加载 OM 模型
B. 转换 ONNX 模型
C. 生成 COCO 标签
D. 安装自定义算子

**解答：** A

**解析：** 该接口用于把 OM 模型加载到 ACL 运行时。


### 问题3（单选题）

**题目：** PyACL 推理中 Host 到 Device 数据拷贝通常发生在哪一步？

A. 输入张量准备好后，模型执行前
B. 模型执行后读取输出时
C. ATC 转换前
D. Git push 后

**解答：** A

**解析：** 输入数据需要先拷贝到 device buffer，模型才能执行。


### 问题4（单选题）

**题目：** 如果脚本输出 `using dry-run output`，最准确的含义是？

A. 已经真实执行 OM 推理
B. 没有真正执行 OM 推理，而是用随机输出帮助流程演示
C. NPU 性能已经最优
D. 自定义算子已经安装

**解答：** B

**解析：** dry-run 是兜底演示，不应作为最终实验结论。


### 问题5（多选题）

**题目：** PyACL 推理脚本中通常需要管理哪些资源？

A. device
B. model_id/model_desc
C. input/output dataset
D. device/host memory buffer

**解答：** A、B、C、D

**解析：** ACL 推理需要显式管理模型、数据集和内存资源。


### 问题6（多选题）

**题目：** 判断 PyACL 真实推理跑通，可以看哪些证据？

A. 打印 loaded OM model
B. used dry-run 为 False
C. 输出 shape 与配置一致
D. postprocess 能得到检测结果或有效输出

**解答：** A、B、C、D

**解析：** 这些证据能说明不是只走了随机兜底路径。


### 问题7（多选题）

**题目：** PyACL 内存管理不当可能导致哪些问题？

A. 内存泄漏
B. 输出读取失败
C. 重复初始化错误
D. 模型执行异常

**解答：** A、B、C、D

**解析：** ACL 编程需要成对释放 dataset、buffer、model 和 device。


### 问题8（判断题）

**题目：** PyACL 加载 OM 后，仍然需要对输入图片做 resize、归一化和 NCHW 排布等预处理。

**解答：** 正确

**解析：** OM 模型只负责网络推理，输入仍需满足模型要求。


### 问题9（判断题）

**题目：** 只要 CPU 后处理能跑通，就能证明 OM 模型一定在 NPU 上执行过。

**解答：** 错误

**解析：** 如果使用 dry-run 输出，CPU 后处理也可能跑通，但不能证明 OM 已真实执行。


### 问题10（填空题）

**题目：** PyACL 中执行模型推理的常用接口是 `acl.mdl.____`。

**解答：** execute

**解析：** acl.mdl.execute 用于执行已加载模型。


### 问题11（填空题）

**题目：** 本实验 PyACL 推理期望的 YOLO 输出 shape 是 `____`。

**解答：** [1, 25200, 85]

**解析：** 该 shape 与 yolo_edge.yaml 中 model.output_shape 一致。


### 问题12（简答题）

**题目：** 为什么 PyACL 推理脚本需要显式释放资源？

**解答：** ACL 编程接近底层运行时，模型描述、dataset、data buffer、device memory 和 device 状态都需要释放。否则反复运行 notebook 时可能出现内存泄漏或设备状态异常。

**解析：** 这也是 Jupyter 反复执行时容易遇到的问题。


### 问题13（简答题）

**题目：** PyACL 推理结果为什么还不能直接等同于最终检测框？

**解答：** YOLO OM 输出是网络预测张量，通常包含大量候选框、目标置信度和类别分数，还需要阈值过滤、坐标转换和 NMS 才能得到最终检测结果。

**解析：** 推理输出与最终检测结果之间还有后处理阶段。


### 问题14（简答题）

**题目：** 如果 PyACL 提示 OM 输出 float32 数量少于配置期望，应该检查什么？

**解答：** 应检查 yolo_edge.yaml 中 output_shape 是否与实际 OM 输出一致，也要确认 ONNX/OM 模型版本和输入 shape 是否匹配。

**解析：** 输出 shape 配置错误会导致读取 buffer 时解释错误。


### 问题15（代码设计题）

**题目：** 写一段最小伪代码，表示 PyACL 推理的资源顺序。

**解答：**

```python
acl.init()
acl.rt.set_device(0)
model_id = acl.mdl.load_from_file("models/yolov5s_310b4.om")[0]
# create desc, input/output datasets, malloc buffers
acl.rt.memcpy(input_dev, input_size, input_host, input_size, ACL_MEMCPY_HOST_TO_DEVICE)
acl.mdl.execute(model_id, input_dataset, output_dataset)
acl.rt.memcpy(output_host, output_size, output_dev, output_size, ACL_MEMCPY_DEVICE_TO_HOST)
# destroy datasets, unload model, reset device, finalize
```

**解析：** 真实代码需检查每个 ret，并做好异常路径释放。
